# Memory Experiment — SHYPS

**[DEMO]** — Build a SHYPS memory with the dedicated cyclic-offset extraction
and inspect its scheduling. The default `r=3` instance is `[[49,9,4]]`.
Each line uses all seven cyclic weight-3 checks: **49 X gauges and 49 Z gauges**
per complete XZ cycle. LightStim generates detectors and logical observables.


In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "lightstim").is_dir() and (path / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.noise.config import NoiseConfig
from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.shyps import SHYPSCode, SHYPSCodeExtractionBlock

In [ ]:
r = 3  # r=4 also works; r=5 is too large for this interactive demo.
basis = "Z"  # "Z" or "X"
rounds = 2  # complete XZ cycles; kept small for the diagram
p = 1e-3
noise = NoiseConfig(p_idle=0, p_1q=0, p_2q=p, p_reset=p, p_meas=p)

## Dedicated scheduling

Each cycle measures X gauges, then Z gauges. Both bases use the following
three offsets in order; reset and readout are additional to the six CNOT layers.

| Instance | CNOT layer 1 | CNOT layer 2 | CNOT layer 3 |
| --- | --- | --- | --- |
| r=3 | 0 | 2 | 3 |
| r=4 | 0 | 1 | 4 |
| r=5 | 0 | 2 | 5 |

For semantic indices `(i,j)` and `m=2**r-1`, layer offset `s` couples
X ancilla `(i,j)` **to** data `((i+s)%m,j)` and data `(i,(j+s)%m)` **to**
Z ancilla `(i,j)`. Each CNOT layer is a matching.

The diagram places the ancilla sectors outside the data grid to distinguish
them. These are display coordinates; the offsets describe connectivity and
gate ordering. Atom movement and wrap-around routing are not compiled here.
The r=5 patch and SE are supported; its full memory construction remains expensive
and has not completed end-to-end validation.


In [ ]:
code = SHYPSCode(r=r)
experiment = MemoryExperiment(
    qec_patch=code,
    extraction_block_class=SHYPSCodeExtractionBlock,
    rounds=rounds,
    basis=basis,
    noise_params=noise,
    noise_model="circuit_level",
)
extraction = SHYPSCodeExtractionBlock(experiment.system)
for gauge_basis, layers in (("X", extraction.x_layers), ("Z", extraction.z_layers)):
    print(f"{gauge_basis}: offsets={code.gauge_offsets}; CNOTs/layer={[len(layer) for layer in layers]}")
circuit = experiment.build()
print(f"Data: {code.num_data_qubits}  Total qubits: {circuit.num_qubits}")
print(f"Detectors: {circuit.num_detectors}  Logical observables: {circuit.num_observables}")

In [ ]:
circuit.without_noise().diagram("detslice-with-ops-svg")

In [ ]:
dem = circuit.detector_error_model()
detectors, observables = circuit.without_noise().compile_detector_sampler(seed=71).sample(
    64, separate_observables=True,
)
assert not detectors.any() and not observables.any()
assert dem.num_observables == r**2
print(f"Noiseless check passed; {r**2} protected logical observables are tracked.")